# Reachy 1.2 — Driving the Arm in the FWD Center Lab

A hands-on tutorial for the **FWDCenterLabMCC** scene: the measured MCC tabletop,
the 3×3 taped grid, and the 80/20 rig frame Reachy is bolted into.

Two routines, written to be read and modified:

1. **Place the arm on the table** — out of the rail pocket and onto the board,
   the motion Siva performs by hand (photos in
   `IITG-Reachy-Project/docs/pics/`).
2. **Raise the arm and move the gripper every way it moves** — one joint at a
   time, then in combination.

Each section says *what* the command does and *why* the numbers are what they
are, then runs it. Change the numbers and re-run — that is the point.

---

### Before you start

Launch the simulator on the lab scene so the physics and the RViz view both show
the real geometry:

```bash
REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh
```

Watch **RViz at http://localhost:6080** and the **cameras at
http://localhost:8080** while cells run.

> **This is the simulator.** On the physical robot every motion must go through
> `src/reachy_ai/motion/primitives.py` with `REACHY_ENABLE_MOTION=true` and a
> human operator present. The angles below are a *teaching* device — once a
> routine settles, promote it to a named pose in `primitives.py`.

## 0. The rail pocket, and why the arm has to reverse out of it

Reachy's torso is bolted to a horizontal 80/20 frame. The right arm hangs down
through an opening in that frame — call it the **pocket**. Measured in the scene:

| | |
|---|---|
| right shoulder | world **(0.000, −0.190, 1.000)** |
| elbow, arm hanging at rest | world **(0.000, −0.190, 0.720)** |
| board, 1 in thick | surface z = **0.740**, underside z = **0.7146** |
| rig rail tops | z = **0.7146** — the board rests **on top of** them |
| board's robot-side edge | x = **0.160** *(still unmeasured)* |
| the pocket | x ∈ [−0.368, **0.160**], y ∈ [−0.304, −0.076] |

The pocket is **9 in wide** (y) and roughly 20 in long (x). The elbow sits
**20 mm below** the rail plane — it is *inside* the slot, not above it.

There is **no cross rail in front of the board**: that edge is finished with a
wooden trim strip overhanging open air (`docs/pics/80903693586`), so the pocket's
forward bound is the board itself, not aluminium.

That geometry dictates the escape:

- **Sideways is the short axis.** Swinging the arm out laterally — *abduction*,
  `r_shoulder_roll` — has only 114 mm before the outer rail. The elbow rises
  just 20 mm in the first 22° of roll, so it hits the rail before it clears it.
  Below −17.5° of roll the hanging arm is already in collision at rest.
- **Backwards is the long axis.** Swinging the arm back along the rail —
  *extension*, `r_shoulder_pitch` **positive**, with roll held at **0** — travels
  the long dimension. At +40° the elbow is at (−0.180, −0.190, **0.786**): it has
  moved 180 mm back, risen 66 mm, and **y has not changed at all**. It is now
  above the rail plane and out of the pocket.

Extension is capped around **+45°** with the arm straight — beyond that the
forearm reaches the rig's *back* rail. +40° is used here because it leaves the
most margin overall: at +45° the forearm comes within 1.7 mm of that back rail,
at +40° it has 44 mm.

Once the arm is clear, curling the forearm tight lifts the whole lower arm high
above the rails, and only then can the shoulder swing the folded unit forward and
out over the front rail onto the board.

> Checked exhaustively: with `r_shoulder_roll` pinned at 0 the arm can reach
> 28 044 configurations but **none** of them puts the hand over the table — the
> elbow crosses the board's near edge at z = 0.770 and the upper-arm capsule is
> 35 mm in radius, so it clips by 5 mm. The roll is needed on the way *in*, not
> on the way *out*. Every waypoint below was verified at 0.2° resolution against
> the board, all five rig rails, the pedestal and the robot itself.

## 1. Connect and take stock

In [ ]:
import time
import sys
import pathlib

# Make src/ importable whether this runs in the container or from the repo.
for _c in (pathlib.Path("/opt/src"), pathlib.Path.cwd().parent / "src"):
    if _c.is_dir() and str(_c) not in sys.path:
        sys.path.insert(0, str(_c))

from reachy_sdk import ReachySDK                 # Reachy 1.2 SDK — NOT reachy2_sdk
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode
from reachy_ai.motion.safety import gate_check
from reachy_ai.scene.awareness import SceneModel

REACHY_HOST = "localhost"
REACHY_PORT = 50051        # fake_reachy_server.py; the physical robot uses 50055

reachy = ReachySDK(host=REACHY_HOST, sdk_port=REACHY_PORT)
arm = reachy.r_arm
print(f"connected to {REACHY_HOST}:{REACHY_PORT}")
print(f"right arm joints: {', '.join(arm.joints.keys())}")
print(f"safety gate_check(): {gate_check()}")

`gate_check()` returns `True` here because `REACHY_SIM_BACKEND` is a simulation
backend. On hardware it returns `False` until `REACHY_ENABLE_MOTION=true` is set
with an operator present. **Never bypass it.**

### The scene, read from the same YAML the simulator loaded

In [ ]:
SCENE_YAML = next(p for p in (
    pathlib.Path("/opt/scenes/FWDCenterLabMCC.yaml"),
    pathlib.Path.cwd().parent / "scenes" / "FWDCenterLabMCC.yaml",
) if p.exists())

scene = SceneModel.from_yaml(str(SCENE_YAML))
table = scene.table

print(f"scene file      : {SCENE_YAML}")
print(f"table surface z : {scene.table_surface_z:.3f} m")
print(f"table extent    : x {table.center[0]-table.size[0]/2:.3f} .. "
      f"{table.center[0]+table.size[0]/2:.3f}   "
      f"y {table.center[1]-table.size[1]/2:.3f} .. {table.center[1]+table.size[1]/2:.3f}")
print(f"rig rails       : {len([o for o in scene.static_obstacles() if 'rig-frame' in o.tags])}")
print()
print("addressable grid cells (robot's view: row 1 = nearest, col 1 = its left):")
for cid in scene.grid_cells():
    x, y, z = scene.cell_center(cid)
    print(f"   {cid}   x={x:+.4f}  y={y:+.4f}  z={z:.4f}")

Two of those nine — **`cell_r3c1` and `cell_r3c2`** — are out of the right arm's
reach (0.709 m and 0.649 m from the shoulder against a 0.609 m maximum). Siva
confirmed that on the physical robot. Plan tasks around the other seven.

## 2. The joint map and its sign conventions

Eight joints, all commanded in **degrees**. The signs are not all intuitive, so
keep this open while you experiment:

| joint | range | what a **positive** value does |
|---|---|---|
| `r_shoulder_pitch` | −150 … +90 | **extension** — swings the arm *backward* along the rail. Negative is **flexion**, forward. |
| `r_shoulder_roll`  | −180 … +10 | negative is **abduction** — swings the arm out sideways, across the rails |
| `r_arm_yaw`        | −90 … +90  | twists the upper arm about its own axis |
| `r_elbow_pitch`    | −125 … 0   | **only negative** — bends the elbow; 0 is a straight arm |
| `r_forearm_yaw`    | −100 … +100| rotates the forearm (pronate / supinate) |
| `r_wrist_pitch`    | −45 … +45  | tilts the hand up / down relative to the forearm |
| `r_wrist_roll`     | −45 … +45  | rolls the hand about the forearm axis |
| `r_gripper`        | −69 … +20  | **inverted: negative OPENS, positive CLOSES** |

Pitch and roll are the two that matter for getting out of the pocket, and they
are *not* interchangeable: pitch moves along the pocket's 19 in axis, roll across
its 9 in axis.

### The gripper sign is backwards from intuition

Measured from `reachy_1_2.xml`, the **right** gripper's pad gap:

| `r_gripper` | pad gap |
|---|---|
| −69° | 7.4 cm (fully open) |
| −45° | 6.5 cm |
| 0°   | 2.6 cm |
| +20° | 0.7 cm (closed) |

The **left** gripper's range is mirrored, so its signs are the opposite way
round. `primitives.open_gripper()` / `close_gripper()` take a `side=` argument
and handle that for you.

In [ ]:
OPEN = -45.0   # gripper open   (~6.5 cm pad gap)
SHUT =  20.0   # gripper closed (~0.7 cm)

R_JOINTS = ["r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw", "r_elbow_pitch",
            "r_forearm_yaw", "r_wrist_pitch", "r_wrist_roll", "r_gripper"]
ARM7 = R_JOINTS[:7]

from reachy_ai.motion.kinematics import CartesianPlanner, R_ARM_JOINTS, UnreachableError

planner = CartesianPlanner(arm, scene, side="right")


def gripper_world_xyz():
    """Gripper pad position in world coordinates, via the SDK's own FK."""
    return planner.fk_world([getattr(arm, j).present_position for j in R_ARM_JOINTS])


def show_pose(label=""):
    print(label)
    print("   " + "  ".join(f"{j[2:]}={getattr(arm, j).present_position:+6.1f}"
                            for j in R_JOINTS))


reachy.turn_on("r_arm")
time.sleep(0.5)
show_pose("arm powered on, current pose:")
print(f"   gripper pad at world {tuple(round(v, 3) for v in gripper_world_xyz())}")

### How to actually command a move

Two things about the MuJoCo physics backend, both learned the hard way:

- **Setting `goal_position` once does nothing.** Under `mujoco-remote` the arm
  only tracks while setpoints are being *streamed*. Holding a goal for 30 s
  leaves the arm exactly where it started.
- **Hand-rolled 25 Hz interpolation tracks badly.** Ramping all joints in a
  Python loop leaves 20–60° of steady-state error — the elbow sags out of its
  fold entirely, which wrecks a route that depends on staying folded.

Use the SDK's own `goto()` with minimum-jerk instead. It tracks to **≈2°** on
every joint. Because minimum jerk applies one shared time profile to all joints,
the *path through joint space is still the straight line* between waypoints —
which is exactly what the collision verification below assumes.

`move_to` also **checks that each waypoint was actually reached** before moving
on, and raises `TrackingError` if not. That matters here: the corridor out of the
pocket is a few millimetres wide, so a waypoint reached 10° short is no longer
the pose that was verified, and continuing from it is how the arm ends up jammed
against a rail. A `TrackingError` is the routine refusing to crash the arm — give
that segment a longer duration and re-run.

In [ ]:
class TrackingError(RuntimeError):
    """The physics arm did not reach a waypoint closely enough to continue."""


# The corridor out of the pocket is only a few millimetres wide, so a waypoint
# that is several degrees short is no longer the pose that was verified.  Rather
# than let that error compound into the next segment (which is how the arm ends
# up jammed against a rail), refuse to continue.
TRACK_TOL = 6.0     # degrees

# Which joints the guard actually polices.  shoulder pitch/roll, arm_yaw, elbow
# and wrist_pitch are what put the elbow, forearm and hand where the clearance
# was measured — those get TRACK_TOL.  r_wrist_roll, r_forearm_yaw and the
# gripper only spin the hand about its own axis; they are weak joints (kp=60,
# 10 Nm) that converge over several waypoints, and a few degrees of error on
# them moves the pad by millimetres inside margins of 20 mm or more.  Holding
# them to 6 deg aborts a route that is in no danger.
CRITICAL = ("r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw",
            "r_elbow_pitch", "r_wrist_pitch")
LOOSE_TOL = 90.0

# Routine 2 runs at PRESENT, in open air ~19 cm above the board with the rig far
# away.  The transit guard is there to stop the arm being driven into the rig;
# it has no job here, and aborting would hide the joint behaviour the lesson
# exists to show.  So the sweeps report their tracking error rather than
# enforcing it.
LESSON_TOL = 90.0


def joint_error(target, critical_only=True):
    names = [n for n in target if n in CRITICAL] if critical_only else \
            [n for n in target if n != "r_gripper"]
    return max((abs(getattr(arm, n).present_position - target[n]), n) for n in names)


def loose_error(target):
    names = [n for n in target
             if n not in CRITICAL and n != "r_gripper"]
    if not names:
        return (0.0, "")
    return max((abs(getattr(arm, n).present_position - target[n]), n) for n in names)


def move_to(label, target, secs, report=True, settle=0.8, tol=TRACK_TOL, retries=3):
    """Interpolate to `target` with the SDK's minimum-jerk trajectory generator.

    A short second pass to the same target re-streams the setpoints and pulls out
    the tracking lag; without it the physics arm can finish a fast segment
    10-20 deg short, and simply *holding* a goal will not close the gap — under
    `mujoco-remote` the arm only moves while setpoints are streaming.
    """
    cmd = {getattr(arm, n): v for n, v in target.items()}
    goto(cmd, duration=secs, interpolation_mode=InterpolationMode.MINIMUM_JERK)
    # Progressively longer settles.  The wrist joints are weak (kp=60,
    # forcerange 10) and a single short pass leaves a large offset half-closed.
    for k in range(retries + 1):
        if settle:
            goto(cmd, duration=settle * (1 + k),
                 interpolation_mode=InterpolationMode.MINIMUM_JERK)
        if joint_error(target)[0] <= tol and loose_error(target)[0] <= LOOSE_TOL:
            break
    err, worst_joint = joint_error(target)
    lerr, ljoint = loose_error(target)
    if lerr > LOOSE_TOL:
        raise TrackingError(
            f"{label}: {ljoint} is {lerr:.1f} deg from its goal (loose tolerance "
            f"{LOOSE_TOL:.0f}).  Even the non-critical joints are not tracking."
        )
    if err > tol:
        raise TrackingError(
            f"{label}: {worst_joint} is {err:.1f} deg from its goal (tolerance "
            f"{tol:.0f}).  The arm is not where the verified route assumes; "
            f"continuing would drive it into the rig.  Re-run the cell, or give "
            f"this segment a longer duration."
        )
    if report:
        x, y, z = gripper_world_xyz()
        print(f"  {label:11s} ({secs:.1f}s)  worst joint error {err:4.1f} deg "
              f"({worst_joint[2:]})   pad -> ({x:+.3f}, {y:+.3f}, {z:.3f})")

## 3. Routine 1 — out of the pocket and onto the board

The motion in four moves, exactly as it is done by hand on the robot:

1. **Back out of the pocket.** `r_shoulder_pitch` → **+40**, roll held at **0**.
   Pure extension along the rail. The elbow travels 180 mm backward and 66 mm up;
   *y never changes*.
2. **Curl the forearm up tight.** Elbow to **−125**, wrist tucked, then extend a
   little further to +70 so the whole folded lower arm rides ~164 mm above the
   rail plane.
3. **Swing the folded unit forward and out over the rail.** Pitch sweeps
   +70 → −17.5 while the roll opens to −37.5, carrying the compact arm over the
   front rail and the table's near edge.
4. **Unfold onto the board.** Elbow opens −120 → −45 and the forearm settles.

| # | pose | pitch / roll | elbow | wrist P/R | grip | elbow world |
|---|---|---|---|---|---|---|
| 0 | `HOME` | 0 / 0 | 0 | 0 / 0 | open | (0.000, −0.190, **0.720**) |
| 1 | `GRIP_SHUT` | 0 / 0 | 0 | 0 / 0 | **shut** | (0.000, −0.190, 0.720) |
| 2 | `BACK` | **+40** / 0 | 0 | 0 / 0 | shut | (−0.180, −0.190, **0.786**) |
| 3 | `CURL` | +40 / 0 | **−125** | +45 / 0 | shut | (−0.180, −0.190, 0.786) |
| 4 | `CURL_HIGH` | **+70** / 0 | −120 | +45 / 0 | shut | (−0.263, −0.190, **0.904**) |
| 5 | `TUCK` | +70 / 0 | −120 | **−45** / 0 | shut | (−0.263, −0.190, 0.904) |
| 6 | `SWING_1` | +37.5 / **−32.5** | −120 | −45 / 0 | shut | (−0.144, −0.340, 0.813) |
| 7 | `SWING_2` | +20 / −35 | −120 | −45 / 0 | shut | (−0.078, −0.351, 0.784) |
| 8 | `SWING_3` | **−17.5** / −37.5 | −120 | −45 / 0 | shut | (+0.067, −0.360, 0.788) |
| 9 | `HOVER` | −40 / −10 | **−60** | −15 / 0 | shut | (+0.177, −0.239, 0.789) |
| 10 | `REST_SHUT` | −40 / −10 | **−45** | −10 / **+30** | shut | (+0.177, −0.239, 0.789) |
| 11 | `REST` | −40 / −10 | −45 | −10 / +30 | **open** | (+0.177, −0.239, 0.789) |

Read the elbow column top to bottom. Rows 0→5 hold **y = −0.190 exactly** — the whole escape happens without a single degree of lateral motion. Only at
row 6, once the elbow is well clear of the rails, does y start to move.

The gripper closes for the transit: an open finger sweeps a wider volume and
catches the front rail. It opens again once the arm is down.

Every segment is contact-free except the last two, which register **−0.97 mm**
against the board. That is the forearm coming to rest — the goal, not a fault.

Measured clearance, worst point in each segment (true surface-to-surface gap):

| segment | gap | closest pair |
|---|---|---|
| HOME → GRIP_SHUT | +79.4 mm | upper arm ↔ inner-right rail |
| GRIP_SHUT → BACK | +44.2 mm | forearm ↔ back rail |
| BACK → CURL | +26.8 mm | thumb pad ↔ board |
| CURL_HIGH → TUCK | +83.9 mm | finger pad ↔ outer-right rail |
| TUCK → SWING_1 | +26.8 mm | forearm ↔ outer-right rail |
| SWING_1 → SWING_2 | +23.4 mm | forearm ↔ board |
| SWING_2 → SWING_3 | +25.9 mm | upper arm ↔ outer-right rail |
| SWING_3 → HOVER | **+4.8 mm** | upper arm ↔ board |

The 4.8 mm is the elbow crossing the board's robot-side edge, and it is the
tightest point on the route. Everything else has centimetres. See §6.

In [ ]:
# ── Verified pose set for FWDCenterLabMCC ────────────────────────────────────
# Checked at 0.2 deg resolution against the board, all five rig rails, the
# pedestal and the robot's own links.  Do not tweak blind — the corridor out of
# the pocket is only a few millimetres wide in places.

def pose(**kw):
    base = dict.fromkeys(ARM7, 0.0)
    base["r_gripper"] = OPEN
    base.update(kw)
    return base


HOME      = pose()
GRIP_SHUT = pose(r_gripper=SHUT)

# 1. back out of the pocket — extension only, roll stays at 0
BACK      = pose(r_gripper=SHUT, r_shoulder_pitch=40.0)

# 2. curl the forearm up tight against the upper arm
CURL      = pose(r_gripper=SHUT, r_shoulder_pitch=40.0,
                 r_elbow_pitch=-125.0, r_wrist_pitch=45.0)
CURL_HIGH = pose(r_gripper=SHUT, r_shoulder_pitch=70.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=45.0)
TUCK      = pose(r_gripper=SHUT, r_shoulder_pitch=70.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)

# 3. swing the folded unit forward and out over the rail
SWING_1   = pose(r_gripper=SHUT, r_shoulder_pitch=37.5, r_shoulder_roll=-32.5,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)
SWING_2   = pose(r_gripper=SHUT, r_shoulder_pitch=20.0, r_shoulder_roll=-35.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)
SWING_3   = pose(r_gripper=SHUT, r_shoulder_pitch=-17.5, r_shoulder_roll=-37.5,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)

# 4. unfold onto the board
HOVER     = pose(r_gripper=SHUT, r_shoulder_pitch=-40.0, r_shoulder_roll=-10.0,
                 r_elbow_pitch=-60.0, r_wrist_pitch=-15.0)
REST_SHUT = pose(r_gripper=SHUT, r_shoulder_pitch=-40.0, r_shoulder_roll=-10.0,
                 r_elbow_pitch=-45.0, r_wrist_pitch=-10.0, r_wrist_roll=30.0)
REST      = dict(REST_SHUT, r_gripper=OPEN)

# Raised pose for routine 2 — every wrist / forearm / gripper angle is
# collision-free from here, so joints can be swept to their limits.
PRESENT   = pose(r_shoulder_pitch=-37.5, r_shoulder_roll=-2.0, r_elbow_pitch=-80.0)

# (name, pose, seconds, tolerance).  Tolerance is per-waypoint because the
# waypoints are not equally dangerous: settling into GRIP_SHUT happens with
# 79 mm of clearance all round and only has to undo the wrist's gravity drift,
# whereas SWING_3 -> HOVER passes the elbow 4.8 mm from the board's edge.
PLACE_ROUTE = [("GRIP_SHUT", GRIP_SHUT, 3.5, 25.0),
               ("BACK",      BACK,      3.0, TRACK_TOL),
               ("CURL",      CURL,      3.0, TRACK_TOL),
               ("CURL_HIGH", CURL_HIGH, 2.0, TRACK_TOL),
               ("TUCK",      TUCK,      2.0, TRACK_TOL),
               ("SWING_1",   SWING_1,   3.0, TRACK_TOL),
               ("SWING_2",   SWING_2,   2.5, TRACK_TOL),
               ("SWING_3",   SWING_3,   2.5, TRACK_TOL),
               ("HOVER",     HOVER,     2.5, TRACK_TOL),
               ("REST_SHUT", REST_SHUT, 3.0, TRACK_TOL),
               ("REST",      REST,      3.0, TRACK_TOL)]

# The stow route is the placement route run backwards.  Nothing may cut across
# it: a direct move from anywhere over the board to HOME drives the upper arm
# through the board's near edge.
STOW_ROUTE = [("REST_SHUT", REST_SHUT, 2.0, TRACK_TOL),
              ("HOVER",     HOVER,     2.0, TRACK_TOL),
              ("SWING_3",   SWING_3,   2.5, TRACK_TOL),
              ("SWING_2",   SWING_2,   2.5, TRACK_TOL),
              ("SWING_1",   SWING_1,   2.0, TRACK_TOL),
              ("TUCK",      TUCK,      3.0, TRACK_TOL),
              ("CURL_HIGH", CURL_HIGH, 2.0, TRACK_TOL),
              ("CURL",      CURL,      2.0, TRACK_TOL),
              ("BACK",      BACK,      3.0, TRACK_TOL),
              ("GRIP_SHUT", GRIP_SHUT, 3.0, 25.0),
              ("HOME",      HOME,      2.5, 25.0)]


# The joints that decide where the arm sits in the rig.  The wrist angles and
# the gripper do not move the elbow or forearm through the rails, and on a
# freshly reset sim they read wherever gravity left them while the motors were
# off (wrist_roll drifts to ~40 deg, the gripper falls open) — so judging "is
# the arm home?" on those would report a clean reset as a fault.
GROSS_JOINTS = ["r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw", "r_elbow_pitch"]


def at_pose(target, tol=8.0, joints=None):
    names = joints if joints is not None else [n for n in target if n != "r_gripper"]
    return all(abs(getattr(arm, n).present_position - target[n]) <= tol for n in names)


def pose_distance(target):
    return max(abs(getattr(arm, n).present_position - v)
               for n, v in target.items() if n != "r_gripper")


def ensure_home():
    """Put the arm back in the pocket before starting a placement.

    Makes the placement cell safe to re-run, and safe to run after an aborted
    one.  Jumping straight to HOME from anywhere over the board would cut the
    corner through the rig's front rail, and so would jumping to the *start* of
    the stow route.  Instead: find the waypoint the arm is already closest to,
    ease onto it, and retrace the route from there.
    """
    if at_pose(HOME, joints=GROSS_JOINTS):
        print("arm already at HOME (wrist/gripper will be set by the first move)\n")
        return
    order = [n for n, _, _, _ in STOW_ROUTE]
    by_name = {n: t for n, t, _, _ in STOW_ROUTE}
    nearest = min(order, key=lambda n: pose_distance(by_name[n]))
    print(f"arm is not at HOME (nearest waypoint: {nearest}, "
          f"{pose_distance(by_name[nearest]):.1f} deg away) — retracing from there")
    try:
        # Ease onto the nearest waypoint slowly, with a loose tolerance: this is
        # the one move on an unverified path, so keep it small and gentle.
        move_to(nearest, by_name[nearest], 4.0, report=False, tol=12.0, retries=3)
        for name, target, secs, tol in STOW_ROUTE[order.index(nearest) + 1:]:
            move_to(name, target, secs, report=False, tol=tol)
    except TrackingError as exc:
        raise TrackingError(
            f"cannot recover to HOME: {exc}\n\n"
            "The arm is most likely WEDGED in the rig — an interrupted routine "
            "can leave the forearm threaded under the front rail, where no "
            "commanded pose will pull it back out (the pad ends up below the "
            "tabletop, z < 0.74).  Reset the simulator rather than fighting it:\n"
            "    REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh\n"
            "then re-run this notebook from the top."
        ) from None
    print(f"   back at HOME (within 8 deg: {at_pose(HOME, joints=GROSS_JOINTS)})\n")


print(f"{len(PLACE_ROUTE)} placement waypoints, {len(STOW_ROUTE)} stow waypoints")

### Running it — about 35 s. Watch RViz.

In [ ]:
reachy.turn_on("r_arm")
time.sleep(0.5)
ensure_home()

print("out of the pocket and onto the board — watch RViz at localhost:6080\n")
for name, target, secs, tol in PLACE_ROUTE:
    move_to(name, target, secs, tol=tol)
    time.sleep(0.3)

print("\narm resting on the board.")
show_pose("final pose:")

### What to try

- **Try the sideways version and watch it fail.** Replace `BACK` with
  `pose(r_gripper=SHUT, r_shoulder_roll=-45.0)` — abduction instead of
  extension. Under the MuJoCo backend the arm jams against the outer rail. That
  jamming *is* the feedback; leave the physics backend on rather than switching
  to `kinematic` to make it look clean.
- **Under-curl the elbow.** Set `CURL_HIGH`'s `r_elbow_pitch` to `-100` and the
  forearm drops back toward the rail plane; `SWING_1` then drives it into the
  outer-right rail.
- **Push the extension too far.** `BACK` at `+45` instead of `+40` brings the
  forearm within 1.7 mm of the rig's *back* rail — still legal, but with no
  margin for the physics arm's tracking error. `+50` fouls it outright.
- **Watch the joint errors.** `move_to` prints the worst residual after each
  waypoint. Anything above ~5° means the physics arm did not get where the plan
  assumed, and the clearance numbers no longer apply — which is why `move_to`
  raises `TrackingError` past 6° rather than carrying on.

> **If a routine is interrupted mid-flight**, the arm can end up wedged: the
> forearm threaded under the board's edge, pad below the surface, where no
> commanded pose pulls it back. `ensure_home()` will tell you so. Reset with
> `REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh` and start again —
> restarting the native server resets the arm's state.

## 4. Routine 2 — raise the arm, then move the gripper every way it moves

Lift out of the rest position to `PRESENT`: gripper pad at
**(0.496, −0.214, 0.948)**, about 21 cm above the board and clear of the rig.
From there **every** angle of `r_wrist_pitch`, `r_wrist_roll`, `r_forearm_yaw`,
`r_arm_yaw` and `r_gripper` is collision-free, so each joint can be driven to its
limits.

In [ ]:
reachy.turn_on("r_arm")
time.sleep(0.3)

move_to("PRESENT", PRESENT, 3.0, tol=LESSON_TOL)
x, y, z = gripper_world_xyz()
print(f"\nraised — pad {z - scene.table_surface_z:.3f} m above the board")

### 4.1 The gripper itself — aperture

One joint, `r_gripper`, sign inverted: **negative opens**.

In [ ]:
print("gripper aperture — negative opens, positive closes\n")
for label, value in [("fully open", -68.0), ("open (working default)", -45.0),
                     ("half",       -20.0), ("nearly shut",           0.0),
                     ("closed",      20.0)]:
    move_to(f"{value:+.0f}", dict(PRESENT, r_gripper=value), 0.9, report=False, tol=LESSON_TOL)
    print(f"  r_gripper = {value:+6.1f}   {label}"
          f"   (present {arm.r_gripper.present_position:+6.1f})")
    time.sleep(0.5)
move_to("open", dict(PRESENT, r_gripper=OPEN), 0.9, report=False, tol=LESSON_TOL)

### 4.2 Wrist pitch — tilt the hand up and down

`r_wrist_pitch`, ±45°. This sets the *approach angle* onto the table: positive
tips the hand up, negative down toward the surface. Its narrow range is why
Reachy 1.2 cannot do a straight top-down grasp — with a near-horizontal forearm,
45° is not enough to point the pads at the floor.

In [ ]:
print("r_wrist_pitch — tilt the hand relative to the forearm (±45°)\n")
for v in (0.0, +45.0, 0.0, -45.0, 0.0):
    move_to(f"wp {v:+.0f}", dict(PRESENT, r_wrist_pitch=v), 1.8, tol=LESSON_TOL)
    time.sleep(0.4)

### 4.3 Wrist roll — rotate the hand about its own axis

`r_wrist_roll`, ±45°. Changes which way the pads face without moving the pad
*position* much. This is how you line the jaws up with an object's long axis.

In [ ]:
print("r_wrist_roll — roll the hand about the forearm axis (±45°)\n")
for v in (0.0, +45.0, 0.0, -45.0, 0.0):
    move_to(f"wr {v:+.0f}", dict(PRESENT, r_wrist_roll=v), 1.8, tol=LESSON_TOL)
    time.sleep(0.4)

### 4.4 Forearm yaw — pronate and supinate

`r_forearm_yaw`, ±100°, the widest of the three. It rotates the whole forearm, so
the hand swings through a much larger arc than `r_wrist_roll` gives. Between the
two you can put the jaws at essentially any roll angle you need.

**This joint does not track well under physics, and the sweep is worth watching
for that reason.** Commanded to a series of angles at `PRESENT` and given 4 s
plus a 2 s settle each, it lands:

| commanded | achieved |
|---|---|
| +15° | −36.8° |
| +45° | −3.6° |
| +90° | +35.0° |
| −45° | −75.1° |
| −90° | −88.3° |

Negative goals track passably; positive ones do not converge at all — the joint
is still swinging when it is sampled. `r_forearm_yaw` has `kp=80` against a
15 Nm limit with `kv=5`, and at this pose that is under-damped. Do not plan a
grasp that depends on a precise positive forearm angle until the actuator gains
are revisited.

Because of that, the sweeps in this section **report** their tracking error
instead of enforcing it — they run in open air ~19 cm above the board, where
there is nothing to hit.

In [ ]:
print("r_forearm_yaw — pronate / supinate (±100°)\n")
# Sweep to +/-90, not the joint's +/-100 hard stop.  At exactly 100 the
# actuator's ctrlrange and the joint limit are the same number, and the physics
# joint sits tens of degrees short of its goal; 90 tracks to a couple of degrees.
for v in (0.0, +90.0, 0.0, -90.0, 0.0):
    move_to(f"fy {v:+.0f}", dict(PRESENT, r_forearm_yaw=v), 3.0, tol=LESSON_TOL)
    time.sleep(0.4)

### 4.5 The joints that *translate* the gripper

The three above mostly change the hand's **orientation**. To move the gripper to
a different **place** you drive the big joints. Note how much further the pad
travels per degree here.

In [ ]:
print("the joints that move the gripper's position\n")
for joint, values in [
    ("r_arm_yaw",        (0.0, -40.0, +40.0, 0.0)),
    ("r_shoulder_roll",  (-2.0, -20.0, +5.0, -2.0)),
    ("r_elbow_pitch",    (-80.0, -95.0, -65.0, -80.0)),
    ("r_shoulder_pitch", (-37.5, -50.0, -25.0, -37.5)),
]:
    print(f"  {joint}:")
    for v in values:
        move_to(f"{v:+.0f}", dict(PRESENT, **{joint: v}), 1.4, tol=LESSON_TOL)
        time.sleep(0.3)

### 4.6 All together — a wave

Multi-joint targets interpolate simultaneously, which is what makes motion look
deliberate rather than sequential.

In [ ]:
WAVE_A = dict(PRESENT, r_forearm_yaw=-60.0, r_wrist_pitch=25.0, r_wrist_roll=-30.0)
WAVE_B = dict(PRESENT, r_forearm_yaw=+60.0, r_wrist_pitch=-25.0, r_wrist_roll=+30.0)

print("combined motion — watch RViz\n")
for i in range(3):
    # Three weak joints reversing together — 0.9 s per swing left them tens of
    # degrees short and tripped the tracking guard.
    move_to(f"wave {i+1}a", WAVE_A, 1.8, report=False, tol=LESSON_TOL)
    move_to(f"wave {i+1}b", WAVE_B, 1.8, report=False, tol=LESSON_TOL)
    print(f"  wave {i + 1}/3")
move_to("PRESENT", PRESENT, 1.8, tol=LESSON_TOL)

### 4.7 Pointing at the grid — from joint angles to world coordinates

`CartesianPlanner` wraps the SDK's IK so you can ask for a **world position**
instead of joint angles. It plans in *pad* space (the contact point between the
jaws, ~0.12 m along the wrist's local −Z) and rejects paths that cross the
tabletop.

Below: hover 12 cm above each grid cell. Two of the nine will fail to solve —
that is the reach limit, reported rather than silently clipped.

In [ ]:
HOVER_H = 0.12   # metres above the board

print(f"hovering {HOVER_H * 100:.0f} cm above each grid cell")
print("(the IK sweeps up to 54 orientations per target — give it a few seconds)\n")
for cid in scene.grid_cells():
    cx, cy, cz = scene.cell_center(cid)
    target = (cx, cy, cz + HOVER_H)
    seed = [getattr(arm, j).present_position for j in R_ARM_JOINTS]
    try:
        q = planner.solve(target, seed=seed)
    except UnreachableError as exc:
        print(f"  {cid}   UNREACHABLE — {exc}")
        continue
    move_to(cid, dict(zip(R_ARM_JOINTS, q)), 1.8, tol=LESSON_TOL)
    time.sleep(0.4)

move_to("PRESENT", PRESENT, 2.0, tol=LESSON_TOL)

## 5. Stow — reverse the route back into the pocket

Retrace the placement route backwards. Do not shortcut it: going straight from
`PRESENT` to `HOME` drives the upper arm through the rig's front rail.

In [ ]:
print("stowing — back into the pocket\n")
for name, target, secs, tol in STOW_ROUTE:
    move_to(name, target, secs, tol=tol)
    time.sleep(0.3)

reachy.turn_off("r_arm")
print(f"\narm at rest in the rail pocket (at HOME: {at_pose(HOME, joints=GROSS_JOINTS)}), motors off.")

## 6. Practice

1. **Re-time the placement.** Halve every duration in `PLACE_ROUTE` and watch the
   residual joint errors `move_to` prints grow. Above ~5° the verified clearances
   stop applying.
2. **Rest the hand elsewhere on the board.** `REST` puts the pad at
   (0.539, −0.249). Shift `r_shoulder_roll`, check with `gripper_world_xyz()`,
   and keep the whole hand inside y ∈ [−0.349, +0.349].
3. **Touch a cell instead of hovering.** Drop `HOVER_H` to 0.02 and see which
   cells still solve.
4. **Write your own routine** as `(name, pose, duration)` tuples, and verify it
   offline before running — the MuJoCo contact check used to build this notebook
   is in `native_mujoco/`.
5. **Promote what works** into `src/reachy_ai/motion/primitives.py`. That is the
   only place motion may come from on the physical robot.

### The open questions this notebook exposes

Two of the three rig numbers `scenes/FWDCenterLabMCC.yaml` used to carry as
*assumed* have been settled from the photographs and the operator, and both were
wrong in the same direction — they put aluminium where there is none:

- **The rails' height.** The board **rests on top of** the frame
  (`docs/pics/80903696209` shows its laminate edge proud of the rail beneath it).
  The rails had been modelled flush with the board's *surface*, putting 25 mm of
  phantom aluminium in the plane the arm has to cross.
- **The front cross rail.** There isn't one. That edge is finished with wooden
  trim overhanging open air (`docs/pics/80903693586`). A `rig_rail_front` had
  been modelled squarely across the arm's route — the single largest obstruction
  in the scene.

Correcting both transformed the swing: its tightest points went from 1.4 mm and
0.3 mm to 23 mm and 4.8 mm.

**One number is still open, and it is now the binding constraint** — the board's
**robot-side edge**, at x = 0.160 in the scene and never measured. The elbow
crosses it at z = 0.770 carrying a 35 mm collision radius, which is the 4.8 mm in
the clearance table above. At x = 0.190 it would clear by 20 mm instead.

Removing the phantom front rail also removed this edge's lower bound, so it is no
longer even bracketed. A related loose end: with no front member the pocket
measures ~20.8 in fore-aft rather than the 19 in the setup notes record — so
either a front member sits further forward than the photos show, or that edge is
closer to the robot than 0.160. One tape measure from the pedestal axis to the
board's near edge settles both.